# Evaluation: LLaMA 3.1-8B for Solana Vulnerability Detection

This notebook evaluates the fine-tuned model across 4 configurations to measure the impact of Fine-Tuning and RAG on vulnerability detection performance.

The 4 configurations:
1. **LLaMA-Base (no RAG)** — baseline, unmodified model
2. **LLaMA-Base + RAG** — base model with retrieval-augmented context
3. **LLaMA-FT (no RAG)** — fine-tuned model, direct prompting
4. **LLaMA-FT + RAG** — fine-tuned model with RAG context

Each configuration is tested on 59 unseen contracts (the test set). We compute Accuracy, Precision, Recall, and F1-Score.

**Reference:** Boi & Esposito (2025), Tortora (2025)

**Hardware:** Kaggle T4 GPU

## 1. Environment Setup

In [ ]:
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install sentence-transformers scikit-learn

## 2. Data Setup

Upload a Kaggle dataset named `solana-evaluation-data` containing these files from your GitHub repo:
- `data/test_set/solana_01.json` ... `solana_59.json`
- `data/knowledge_base/vulnerability_info.json`
- `data/knowledge_base/rag_contracts.jsonl`

After adding the dataset via "Add Input", the files will be accessible under `/kaggle/input/`.

In [ ]:
import os, json, glob

# Find the input path dynamically
input_base = "/kaggle/input"
all_files = []
for root, dirs, files in os.walk(input_base):
    for f in files:
        all_files.append(os.path.join(root, f))

# Locate key files
test_files = sorted([f for f in all_files if "solana_" in f and f.endswith(".json")])
vuln_info_file = [f for f in all_files if "vulnerability_info" in f]
rag_contracts_file = [f for f in all_files if "rag_contracts" in f]

print(f"Test contracts found: {len(test_files)}")
print(f"Vulnerability info: {vuln_info_file[0] if vuln_info_file else 'NOT FOUND'}")
print(f"RAG contracts: {rag_contracts_file[0] if rag_contracts_file else 'NOT FOUND'}")

if len(test_files) != 59:
    print("WARNING: Expected 59 test files")

VULN_INFO_PATH = vuln_info_file[0]
RAG_CONTRACTS_PATH = rag_contracts_file[0]

## 3. Define Evaluation Prompts

Two prompt sets are needed:
- **No-RAG prompt**: asks the model to analyze code directly
- **RAG prompt**: includes a vulnerability checklist retrieved from the knowledge base

Both prompts instruct the model to use `<think>` for reasoning and `<final>` for the verdict, matching the training format.

In [ ]:
# System prompt for direct analysis (no RAG)
SYSTEM_PROMPT_NO_RAG = """You are an expert smart contract security auditor specialized in the Solana blockchain and Rust.
Your task is to analyze Rust code precisely and systematically to identify security vulnerabilities.

Required behavior:
- Perform a structured reasoning phase inside <think>...</think> tags.
- After </think>, provide the final judgment inside <final>...</final> tags with this structure:
  - A short summary sentence.
  - ### Vulnerability: <name or "Not Vulnerable">
  - ### Explanation: <concise cause and how it can be exploited>
  - ### Risk: <severity (Critical/High/Medium/Low) and short impact statement>

If no vulnerability is found, write "### Vulnerability: Not Vulnerable".
Do NOT include any extra commentary outside these tags.""".strip()

USER_PROMPT_NO_RAG = """Please perform a detailed security analysis of the following Solana smart contract.
Carefully examine its logic, identify any potential vulnerabilities:

Contract Code:

```rust
{code}
```""".strip()

# System prompt for RAG-augmented analysis
SYSTEM_PROMPT_RAG = """You are an expert smart contract security auditor specialized in the Solana blockchain and Rust.
Your task is to analyze Rust code precisely and systematically to identify security vulnerabilities, leveraging additional contextual information retrieved via RAG.

Required behavior:
- Perform a structured reasoning phase inside <think>...</think> tags.
  - For each RAG-suggested vulnerability, check whether the code enforces the required security invariant.
  - Also remain open to detecting other vulnerabilities not present in the RAG list.
- After </think>, provide the final judgment inside <final>...</final> tags with this structure:
  - A short summary sentence.
  - ### Vulnerability: <name or "Not Vulnerable">
  - ### Explanation: <concise cause and how it could be exploited>
  - ### Risk: <severity (Critical/High/Medium/Low) and short impact statement>

If no vulnerability is found, write "### Vulnerability: Not Vulnerable".
Do NOT include any extra commentary outside these tags.""".strip()

USER_PROMPT_RAG = """Here is a vulnerability checklist (from RAG) with priority guidance. Use it to steer your audit, but also look for issues beyond this list.

{checklist}

Now perform a detailed security analysis of the following Solana smart contract:

Contract Code:

```rust
{code}
```""".strip()

# System prompt for generating code descriptions (used by RAG retrieval)
SYSTEM_PROMPT_DESCRIPTION = """You are an expert smart contract security auditor specialized in Solana and Rust.
Describe the given Solana program with exactly these 3 sections:

### Contract type:
State whether this is a "Solana program instruction handler".

### Contract Purpose
One short paragraph stating the primary objective.

### Functional Description
Step-by-step description of the execution logic. Be explicit about which accounts are validated and how.
Do not include variable names or code syntax.""".strip()

print("Prompts defined.")

## 4. Response Parser

Extracts the vulnerability verdict from the model's `<final>` block. The parser recognizes all 7 Solana vulnerability types plus "Not Vulnerable". If the model fails to produce valid tags, we retry up to 3 times.

In [ ]:
import re

KNOWN_VULNERABILITIES = [
    "Missing Key Check", "Access Control",
    "Type Confusion", "Input Validation",
    "CPI Reentrancy", "Reentrancy",
    "Unchecked External Calls", "Unchecked Calls",
    "Integer Overflow", "Integer Underflow", "Integer Flow",
    "Bump Seed", "Bump Seed Canonicalization",
    "Denial of Service", "DoS",
    "Not Vulnerable", "No security risk", "No vulnerabilities", "None detected",
]

def parse_response(response):
    """Extract vulnerability verdict from the <final> block.
    Returns (success, list_of_vulnerabilities)."""
    match = re.search(r"<final>\s*(.*?)\s*</final>", response, re.IGNORECASE | re.DOTALL)
    if not match:
        return False, []

    content = match.group(1).strip().lower()
    found = [v for v in KNOWN_VULNERABILITIES if v.lower() in content]

    return (True, found) if found else (False, [])

# Mapping from test file vulnerability keys to display names
VULN_DISPLAY = {
    "missing_key_check": "Missing Key Check",
    "type_confusion": "Type Confusion",
    "cpi_reentrancy": "CPI Reentrancy",
    "unchecked_calls": "Unchecked External Calls",
    "integer_overflow": "Integer Overflow",
    "bump_seed": "Bump Seed",
    "dos": "Denial of Service",
    "not_vulnerable": "Not Vulnerable",
}

print("Parser ready.")

## 5. RAG Pipeline

The RAG system works in 3 steps:
1. Generate a security-focused description of the target contract
2. Find the most similar contracts in the knowledge base using cosine similarity
3. Build a prioritized checklist to inject into the audit prompt

We use the `hkunlp/instructor-xl` embedding model, the same one used during knowledge base construction.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the embedding model
print("Loading embedding model (this takes about 1 minute)...")
embed_model = SentenceTransformer("hkunlp/instructor-xl")
print("Embedding model loaded.")

# Load the vulnerability knowledge base
with open(VULN_INFO_PATH, "r", encoding="utf-8") as f:
    vuln_info = json.load(f)

# Load and embed the RAG contracts
rag_contracts = []
with open(RAG_CONTRACTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rag_contracts.append(json.loads(line))

print(f"RAG knowledge base: {len(rag_contracts)} contracts, {len(vuln_info)} vulnerability types")

# Pre-compute embeddings for all RAG contracts
instruction = "Represent the following functional description of a Solana smart contract, written in Rust:"
rag_embeddings = []
for contract in rag_contracts:
    emb = embed_model.encode([instruction, contract["description"]])
    rag_embeddings.append(emb)
rag_embeddings = np.array(rag_embeddings)

print(f"Embeddings computed: {rag_embeddings.shape}")


def search_similar_contracts(query_description, top_k=None):
    """Find the most similar contracts in the knowledge base.
    Returns one result per vulnerability type, sorted by similarity."""
    query_emb = embed_model.encode([instruction, query_description]).reshape(1, -1)

    results = []
    for i, emb in enumerate(rag_embeddings):
        sim = cosine_similarity(emb.reshape(1, -1), query_emb)[0][0]
        results.append({
            "vulnerability": rag_contracts[i]["vulnerability"],
            "vulnerable_part": rag_contracts[i]["vulnerable_part"],
            "similarity": f"{sim:.4f}",
        })

    # Sort by similarity descending
    results.sort(key=lambda x: x["similarity"], reverse=True)

    # Keep only the best match per vulnerability type
    seen = set()
    unique_results = []
    for r in results:
        if r["vulnerability"] not in seen:
            seen.add(r["vulnerability"])
            unique_results.append(r)

    return unique_results


def build_rag_checklist(similar_contracts, num_detailed=3):
    """Build a prioritized vulnerability checklist from retrieval results."""
    checklist = ""
    for i, contract in enumerate(similar_contracts):
        vuln_key = contract["vulnerability"]
        if vuln_key not in vuln_info:
            continue

        info = vuln_info[vuln_key]
        score = contract["similarity"]

        if i < num_detailed:
            checklist += f"""
    {i+1}. **{info['name']}** ({score})
        - Description: {info['description']}
        - Preconditions: {info['precondition']}
        - Security check: {info['security_check']}
        - Example vulnerable pattern:
          ```rust
    {contract['vulnerable_part'].strip()}
          ```
"""
        else:
            checklist += f"""
    {i+1}. **{info['name']}** ({score})
"""
    return checklist

print("RAG pipeline ready.")

## 6. Load Models

We load two versions of the model:
- **Base model**: the original LLaMA 3.1-8B-Instruct (no fine-tuning)
- **Fine-tuned model**: the LoRA adapter trained in the previous notebook, loaded from Hugging Face Hub

Both are loaded in 4-bit quantization to fit on a single T4 GPU. We evaluate one at a time to manage memory.

In [ ]:
from unsloth import FastLanguageModel
import torch, gc

def load_base_model():
    """Load the original LLaMA 3.1-8B-Instruct without fine-tuning."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def load_finetuned_model():
    """Load the fine-tuned model from Hugging Face Hub."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="Mustafa99Hafed/LLaMA-3.1-8B-Solana-Audit",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def unload_model(model, tokenizer):
    """Free GPU memory before loading the next model."""
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

def generate_response(model, tokenizer, messages, max_tokens=1024):
    """Generate a response from the model given a list of messages."""
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=max_tokens, temperature=0.6, top_p=0.95
    )
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("Model loading functions ready.")

## 7. Run All 4 Configurations

For each configuration, we run the model on all 59 test contracts and record the results. Each test contract is analyzed up to 3 times if the model fails to produce valid `<final>` tags.

This is the main evaluation cell and takes the longest to run (approximately 2-3 hours total for all 4 configurations).

In [ ]:
from collections import defaultdict

# Vulnerability aliases - synonyms that refer to the same vulnerability
VULNERABILITY_ALIASES = {
    "Missing Key Check": ["missing key check", "access control"],
    "Type Confusion": ["type confusion", "input validation"],
    "CPI Reentrancy": ["cpi reentrancy", "cpi", "reentrancy"],
    "Unchecked External Calls": ["unchecked external calls", "unchecked calls"],
    "Integer Overflow": ["integer overflow", "integer underflow", "integer flow"],
    "Bump Seed": ["bump seed", "bump seed canonicalization"],
    "Denial of Service": ["denial of service", "dos"],
    "Not Vulnerable": ["not vulnerable", "no security risk", "no vulnerabilities", "none detected"],
}

GROUND_TRUTH_MAP = {
    "integer_flow": "Integer Overflow",
    "cpi": "CPI Reentrancy",
}


def normalize_predictions(predictions):
    """Map raw predictions to canonical vulnerability names, merging synonyms."""
    canonical_set = set()
    for pred in predictions:
        pred_lower = pred.lower().strip()
        for canonical, aliases in VULNERABILITY_ALIASES.items():
            if pred_lower in aliases:
                canonical_set.add(canonical)
                break
    return canonical_set


def evaluate_config(model, tokenizer, test_files, use_rag=False, config_name=""):
    """Run evaluation on all test contracts for one configuration."""
    results = []
    total = len(test_files)

    for idx, test_file in enumerate(test_files):
        with open(test_file, "r", encoding="utf-8") as f:
            test_data = json.load(f)

        code = test_data["smart_contract"]
        ground_truth = test_data["vulnerability"]
        truth_display = VULN_DISPLAY.get(ground_truth, ground_truth)

        if use_rag:
            desc_messages = [
                {"role": "system", "content": SYSTEM_PROMPT_DESCRIPTION},
                {"role": "user", "content": f"Describe this Solana contract:\n\n```rust\n{code}\n```"},
            ]
            description = generate_response(model, tokenizer, desc_messages, max_tokens=512)
            similar = search_similar_contracts(description)
            checklist = build_rag_checklist(similar)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT_RAG},
                {"role": "user", "content": USER_PROMPT_RAG.format(checklist=checklist, code=code)},
            ]
        else:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT_NO_RAG},
                {"role": "user", "content": USER_PROMPT_NO_RAG.format(code=code)},
            ]

        found_vulns = []
        for attempt in range(3):
            response = generate_response(model, tokenizer, messages)
            success, found_vulns = parse_response(response)
            if success:
                break

        results.append({
            "file": os.path.basename(test_file),
            "ground_truth": truth_display,
            "predictions": found_vulns,
            "raw_response": response,
        })

        print(f"  [{idx+1}/{total}] {os.path.basename(test_file)} | Truth: {truth_display} | Predicted: {found_vulns}")

    return results


def compute_metrics(results):
    """Compute Accuracy, Precision, Recall, F1 with proper alias handling."""
    tp = fp = tn = fn = 0
    per_vuln = defaultdict(lambda: {"tp": 0, "fp": 0, "tn": 0, "fn": 0})

    for r in results:
        truth = GROUND_TRUTH_MAP.get(r["ground_truth"], r["ground_truth"])
        predictions = normalize_predictions(r["predictions"])

        if not predictions:
            if truth == "Not Vulnerable":
                tn += 1
                per_vuln[truth]["tn"] += 1
            else:
                fn += 1
                per_vuln[truth]["fn"] += 1
            continue

        if truth in predictions:
            if truth == "Not Vulnerable":
                tn += 1
                per_vuln[truth]["tn"] += 1
                extras = predictions - {"Not Vulnerable"}
                fp += len(extras)
                for e in extras:
                    per_vuln[e]["fp"] += 1
            else:
                tp += 1
                per_vuln[truth]["tp"] += 1
                extras = predictions - {truth, "Not Vulnerable"}
                fp += len(extras)
                for e in extras:
                    per_vuln[e]["fp"] += 1
        else:
            if truth == "Not Vulnerable":
                extras = predictions - {"Not Vulnerable"}
                fp += len(extras)
                for e in extras:
                    per_vuln[e]["fp"] += 1
            else:
                fn += 1
                per_vuln[truth]["fn"] += 1
                wrong = predictions - {"Not Vulnerable"}
                fp += len(wrong)
                for e in wrong:
                    per_vuln[e]["fp"] += 1

    total = tp + fp + tn + fn
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / total if total > 0 else 0

    return {
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "confusion_matrix": {"tp": tp, "fp": fp, "tn": tn, "fn": fn},
        "per_vulnerability": dict(per_vuln),
    }


print("Evaluation functions ready (with alias handling).")

## 8. Configurations 1 & 2: Base Model

First we evaluate the unmodified LLaMA 3.1-8B-Instruct, both without and with RAG.

In [ ]:
# Load the base model
print("Loading base model...")
model_base, tokenizer_base = load_base_model()
print("Base model loaded.\n")

# Config 1: Base, no RAG
print("=" * 60)
print("CONFIG 1: LLaMA-Base (no RAG)")
print("=" * 60)
results_base_no_rag = evaluate_config(model_base, tokenizer_base, test_files, use_rag=False, config_name="Base-NoRAG")
metrics_base_no_rag = compute_metrics(results_base_no_rag)
print(f"\nResults: {metrics_base_no_rag}\n")

# Config 2: Base, with RAG
print("=" * 60)
print("CONFIG 2: LLaMA-Base + RAG")
print("=" * 60)
results_base_rag = evaluate_config(model_base, tokenizer_base, test_files, use_rag=True, config_name="Base-RAG")
metrics_base_rag = compute_metrics(results_base_rag)
print(f"\nResults: {metrics_base_rag}\n")

# Free memory for the next model
unload_model(model_base, tokenizer_base)
print("Base model unloaded.")

## 9. Configurations 3 & 4: Fine-Tuned Model

Now we evaluate the fine-tuned model (loaded from Hugging Face Hub).

In [ ]:
from huggingface_hub import YOUR_HF_TOKEN_HERE_download, HfApi

config_path = YOUR_HF_TOKEN_HERE_download("unsloth/Meta-Llama-3.1-8B-Instruct", "config.json")

api = HfApi(token="YOUR_HF_TOKEN_HERE")
api.upload_file(
    path_or_fileobj=config_path,
    path_in_repo="config.json",
    repo_id="Mustafa99Hafed/LLaMA-3.1-8B-Solana-Audit",
)
print("Done!")

In [ ]:
# Load the fine-tuned model
print("Loading fine-tuned model from Hugging Face...")
model_ft, tokenizer_ft = load_finetuned_model()
print("Fine-tuned model loaded.\n")

# Config 3: Fine-tuned, no RAG
print("=" * 60)
print("CONFIG 3: LLaMA-FT (no RAG)")
print("=" * 60)
results_ft_no_rag = evaluate_config(model_ft, tokenizer_ft, test_files, use_rag=False, config_name="FT-NoRAG")
metrics_ft_no_rag = compute_metrics(results_ft_no_rag)
print(f"\nResults: {metrics_ft_no_rag}\n")

# Config 4: Fine-tuned, with RAG
print("=" * 60)
print("CONFIG 4: LLaMA-FT + RAG")
print("=" * 60)
results_ft_rag = evaluate_config(model_ft, tokenizer_ft, test_files, use_rag=True, config_name="FT-RAG")
metrics_ft_rag = compute_metrics(results_ft_rag)
print(f"\nResults: {metrics_ft_rag}\n")

unload_model(model_ft, tokenizer_ft)
print("Fine-tuned model unloaded.")

## 10. Results Summary

Comparison table across all 4 configurations, showing the impact of Fine-Tuning and RAG on detection performance.

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Configuration": "LLaMA-Base (no RAG)",
        "Accuracy": metrics_base_no_rag["accuracy"],
        "Precision": metrics_base_no_rag["precision"],
        "Recall": metrics_base_no_rag["recall"],
        "F1-Score": metrics_base_no_rag["f1_score"],
        "TP": metrics_base_no_rag["confusion_matrix"]["tp"],
        "FP": metrics_base_no_rag["confusion_matrix"]["fp"],
        "TN": metrics_base_no_rag["confusion_matrix"]["tn"],
        "FN": metrics_base_no_rag["confusion_matrix"]["fn"],
    },
    {
        "Configuration": "LLaMA-Base + RAG",
        "Accuracy": metrics_base_rag["accuracy"],
        "Precision": metrics_base_rag["precision"],
        "Recall": metrics_base_rag["recall"],
        "F1-Score": metrics_base_rag["f1_score"],
        "TP": metrics_base_rag["confusion_matrix"]["tp"],
        "FP": metrics_base_rag["confusion_matrix"]["fp"],
        "TN": metrics_base_rag["confusion_matrix"]["tn"],
        "FN": metrics_base_rag["confusion_matrix"]["fn"],
    },
    {
        "Configuration": "LLaMA-FT (no RAG)",
        "Accuracy": metrics_ft_no_rag["accuracy"],
        "Precision": metrics_ft_no_rag["precision"],
        "Recall": metrics_ft_no_rag["recall"],
        "F1-Score": metrics_ft_no_rag["f1_score"],
        "TP": metrics_ft_no_rag["confusion_matrix"]["tp"],
        "FP": metrics_ft_no_rag["confusion_matrix"]["fp"],
        "TN": metrics_ft_no_rag["confusion_matrix"]["tn"],
        "FN": metrics_ft_no_rag["confusion_matrix"]["fn"],
    },
    {
        "Configuration": "LLaMA-FT + RAG",
        "Accuracy": metrics_ft_rag["accuracy"],
        "Precision": metrics_ft_rag["precision"],
        "Recall": metrics_ft_rag["recall"],
        "F1-Score": metrics_ft_rag["f1_score"],
        "TP": metrics_ft_rag["confusion_matrix"]["tp"],
        "FP": metrics_ft_rag["confusion_matrix"]["fp"],
        "TN": metrics_ft_rag["confusion_matrix"]["tn"],
        "FN": metrics_ft_rag["confusion_matrix"]["fn"],
    },
])

print(summary.to_string(index=False))

# Save results to JSON
all_results = {
    "base_no_rag": metrics_base_no_rag,
    "base_rag": metrics_base_rag,
    "ft_no_rag": metrics_ft_no_rag,
    "ft_rag": metrics_ft_rag,
}

with open("/kaggle/working/evaluation_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("\nResults saved to evaluation_results.json")

## 11. Visualization

Bar charts comparing the 4 configurations across all metrics, following the same visual style used by Tortora (2025) in the reference thesis.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

configs = ["Base\n(no RAG)", "Base\n+ RAG", "FT\n(no RAG)", "FT\n+ RAG"]
metrics_list = {
    "Precision": [
        metrics_base_no_rag["precision"], metrics_base_rag["precision"],
        metrics_ft_no_rag["precision"], metrics_ft_rag["precision"],
    ],
    "Recall": [
        metrics_base_no_rag["recall"], metrics_base_rag["recall"],
        metrics_ft_no_rag["recall"], metrics_ft_rag["recall"],
    ],
    "F1-Score": [
        metrics_base_no_rag["f1_score"], metrics_base_rag["f1_score"],
        metrics_ft_no_rag["f1_score"], metrics_ft_rag["f1_score"],
    ],
    "Accuracy": [
        metrics_base_no_rag["accuracy"], metrics_base_rag["accuracy"],
        metrics_ft_no_rag["accuracy"], metrics_ft_rag["accuracy"],
    ],
}

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
colors = ["#5B9BD5", "#70AD47", "#FFC000", "#ED7D31"]

for ax, (metric_name, values) in zip(axes, metrics_list.items()):
    bars = ax.bar(configs, values, color=colors, edgecolor="white", width=0.6)
    ax.set_title(metric_name, fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score" if metric_name == "Precision" else "")
    ax.grid(axis="y", alpha=0.3)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.2f}", ha="center", va="bottom", fontsize=10)

fig.suptitle("Impact of Fine-Tuning and RAG on Vulnerability Detection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/evaluation_chart.png", dpi=150, bbox_inches="tight")
plt.show()

print("Chart saved to evaluation_chart.png")

## 12. Fine-Tuning Impact Analysis

Side-by-side comparison of Base vs Fine-Tuned model (without RAG) to isolate the contribution of fine-tuning alone.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

metric_names = ["Precision", "Recall", "F1-Score", "Accuracy"]
base_values = [
    metrics_base_no_rag["precision"], metrics_base_no_rag["recall"],
    metrics_base_no_rag["f1_score"], metrics_base_no_rag["accuracy"],
]
ft_values = [
    metrics_ft_no_rag["precision"], metrics_ft_no_rag["recall"],
    metrics_ft_no_rag["f1_score"], metrics_ft_no_rag["accuracy"],
]

x = np.arange(len(metric_names))
width = 0.3

bars1 = ax.bar(x - width/2, base_values, width, label="LLaMA-Base", color="#5B9BD5", edgecolor="white")
bars2 = ax.bar(x + width/2, ft_values, width, label="LLaMA-FT", color="#70AD47", edgecolor="white")

ax.set_ylabel("Score (0-1)")
ax.set_title("Impact of Fine-Tuning on Performance (No-RAG)", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("/kaggle/working/finetuning_impact.png", dpi=150, bbox_inches="tight")
plt.show()

print("Chart saved to finetuning_impact.png")

## 13. RAG Impact Analysis

Comparison showing how RAG affects the fine-tuned model's performance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ft_no_rag_values = [
    metrics_ft_no_rag["precision"], metrics_ft_no_rag["recall"],
    metrics_ft_no_rag["f1_score"], metrics_ft_no_rag["accuracy"],
]
ft_rag_values = [
    metrics_ft_rag["precision"], metrics_ft_rag["recall"],
    metrics_ft_rag["f1_score"], metrics_ft_rag["accuracy"],
]

bars1 = ax.bar(x - width/2, ft_no_rag_values, width, label="FT (no RAG)", color="#5B9BD5", edgecolor="white")
bars2 = ax.bar(x + width/2, ft_rag_values, width, label="FT + RAG", color="#70AD47", edgecolor="white")

ax.set_ylabel("Score (0-1)")
ax.set_title("Impact of RAG on Fine-Tuned Model", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("/kaggle/working/rag_impact.png", dpi=150, bbox_inches="tight")
plt.show()

print("Chart saved to rag_impact.png")

## 14. Export All Results

Package the evaluation results, charts, and detailed logs for download.

In [ ]:
import shutil

# Save detailed results per configuration
detailed = {
    "base_no_rag": [{"file": r["file"], "truth": r["ground_truth"], "predicted": r["predictions"]} for r in results_base_no_rag],
    "base_rag": [{"file": r["file"], "truth": r["ground_truth"], "predicted": r["predictions"]} for r in results_base_rag],
    "ft_no_rag": [{"file": r["file"], "truth": r["ground_truth"], "predicted": r["predictions"]} for r in results_ft_no_rag],
    "ft_rag": [{"file": r["file"], "truth": r["ground_truth"], "predicted": r["predictions"]} for r in results_ft_rag],
}

with open("/kaggle/working/detailed_results.json", "w") as f:
    json.dump(detailed, f, indent=2)

print("All outputs in /kaggle/working/:")
for f in sorted(os.listdir("/kaggle/working")):
    if not f.startswith("."):
        size = os.path.getsize(f"/kaggle/working/{f}")
        print(f"  {f} ({size // 1024}KB)")

print("\nDownload these files from the Output tab.")